# De una entrada a una predicción
**Estud-IA · Capa 2 · IA nivel intermedio · Encuentro 2 · Ruta machine learning (Python)**

Hoy construimos una **red neuronal muy pequeña** para ver qué ocurre dentro de un paso de aprendizaje: recorrido hacia adelante, pérdida, gradientes y optimizador. Después leemos la curva de entrenamiento y validación **antes de explicarla**.

**Cómo se usa:** ejecuta cada celda en orden con el botón ▶ (o `Shift + Enter`). En las celdas marcadas con 🔮, **escribe en el chat qué esperas** antes de ejecutar.

> Todos los mensajes son **ficticios** (una tienda en línea inventada). Si tu proyecto es de **visión**, la lógica es la misma: cambia el texto por imágenes y la capa de texto por una convolucional.

## Paso 0 · Preparar el entorno
**Usa CPU, no GPU:** la red es diminuta. En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU*.

TensorFlow y Keras ya vienen instalados en Colab.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

keras.utils.set_random_seed(42)   # para que todos veamos casi lo mismo
print("TensorFlow", tf.__version__, "· Keras", keras.__version__)

## Paso 1 · Los datos
Mensajes ficticios de clientes con tres motivos: **pagos**, **envíos** y **acceso**. Los generamos a partir de frases base con pequeñas variaciones.

Los separamos en tres grupos, y **cada uno tiene un papel**:
- **Entrenamiento:** de aquí aprende el modelo.
- **Validación:** con esto decidimos si lo aprendido sirve con casos que no vio.
- **Reservado (prueba):** **no lo tocamos hoy.** Se reserva para comprobar al final, no para elegir ajustes.

In [ ]:
PLANTILLAS = {
    "pagos": ['me cobraron dos veces el pedido', 'mi pago con tarjeta no pasó', 'necesito la factura de mi compra', 'el cobro no coincide con el precio', 'quiero que me devuelvan el dinero', 'me descontaron de más al pagar', 'el reembolso todavía no aparece', 'no entiendo un cargo en mi tarjeta'],
    "envios": ['mi pedido no ha llegado', 'quiero cambiar mi dirección de envío', 'el paquete llegó con retraso', 'dónde está mi pedido', 'la guía de envío no avanza', 'me entregaron otro paquete', 'el repartidor no encontró mi casa', 'mi paquete llegó abierto'],
    "acceso": ['no puedo entrar a mi cuenta', 'olvidé mi contraseña', 'el código de verificación no llega', 'mi usuario aparece bloqueado', 'no me deja iniciar sesión', 'la aplicación cierra mi sesión sola', 'no recibo el correo para restablecer la clave', 'mi cuenta dice que no existe'],
}

PRE = ['', 'hola, ', 'buenas tardes, ', 'por favor, ', 'urgente: ']
SUF = ['', ' gracias', ' ayuda por favor', ' desde ayer']
CATEGORIAS = list(PLANTILLAS.keys())

def generar(n_por_clase=100, semilla=7):
    rnd = random.Random(semilla)
    X, y = [], []
    for k, (cat, frases) in enumerate(PLANTILLAS.items()):
        vistos = set()
        while len(vistos) < n_por_clase:
            vistos.add(rnd.choice(PRE) + rnd.choice(frases) + rnd.choice(SUF))
        for t in sorted(vistos):
            X.append(t); y.append(k)
    orden = list(range(len(X)))
    rnd.shuffle(orden)
    return [X[i] for i in orden], np.array([y[i] for i in orden])

X, y = generar()
X_entrena, y_entrena = X[:210], y[:210]
X_valida,  y_valida  = X[210:255], y[210:255]
X_reservado, y_reservado = X[255:], y[255:]    # no lo usamos hoy

print("Entrenamiento:", len(X_entrena), "| Validación:", len(X_valida), "| Reservado:", len(X_reservado))
for t, k in list(zip(X_entrena, y_entrena))[:5]:
    print(f"  {CATEGORIAS[k]:<7} ← {t}")

## Paso 2 · Construye el modelo y míralo por dentro
Un modelo es **entrada → capas → pesos → salida**:

1. `TextVectorization`: convierte cada texto en una lista de números (ids de palabras).
2. `Embedding`: convierte cada id en un vector de 16 números.
3. `GlobalAveragePooling1D`: promedia los vectores de las palabras del mensaje.
4. `Dense` (oculta): mezcla esos números.
5. `Dense` (salida): una probabilidad por categoría.

In [ ]:
def construir_modelo(unidades=16):
    vectorizador = keras.layers.TextVectorization(max_tokens=300, output_sequence_length=12)
    vectorizador.adapt(tf.constant(X_entrena))          # aprende el vocabulario SOLO con entrenamiento
    modelo = keras.Sequential([
        keras.Input(shape=(1,), dtype="string"),
        vectorizador,
        keras.layers.Embedding(300, 16),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(unidades, activation="relu"),
        keras.layers.Dense(len(CATEGORIAS), activation="softmax"),
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(0.01),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

modelo = construir_modelo()
modelo.summary()

🔮 **Antes de la siguiente celda:** el modelo **todavía no aprendió nada** (sus pesos son aleatorios). Con tres categorías, ¿qué probabilidades esperas para cada mensaje? Escríbelo en el chat.

In [ ]:
mensajes_prueba = ["me cobraron dos veces el pedido", "mi pedido no ha llegado", "olvidé mi contraseña"]
probas = modelo.predict(tf.constant(mensajes_prueba), verbose=0)

for t, p in zip(mensajes_prueba, probas):
    print(t)
    print("   ", {c: round(float(v), 2) for c, v in zip(CATEGORIAS, p)})

## Paso 3 · Un solo paso de aprendizaje, por dentro
`fit` hace esto muchas veces. Aquí lo hacemos **una vez, a mano**, para ver las cuatro partes:

1. **Recorrido hacia adelante:** el lote de mensajes pasa por las capas y sale una predicción.
2. **Pérdida:** mide qué tan lejos quedó la predicción de la etiqueta.
3. **Gradientes:** indican hacia dónde mover cada peso para reducir esa pérdida.
4. **Optimizador:** actualiza los pesos.

🔮 Después de dar el paso, ¿la pérdida debería **subir** o **bajar**?

In [ ]:
lote_x = tf.constant(X_entrena[:32])
lote_y = tf.constant(y_entrena[:32])
perdida_fn = keras.losses.SparseCategoricalCrossentropy()
optimizador = keras.optimizers.Adam(0.01)

# 1) Recorrido hacia adelante  y  2) Pérdida
with tf.GradientTape() as cinta:
    predicciones = modelo(lote_x, training=True)
    perdida = perdida_fn(lote_y, predicciones)
print(f"Pérdida ANTES del paso: {float(perdida):.4f}")

# 3) Gradientes
variables = modelo.trainable_variables
gradientes = cinta.gradient(perdida, variables)
print("Tamaño del gradiente en la última capa:", float(tf.norm(gradientes[-2])))

# 4) Optimizador: mueve los pesos
optimizador.apply_gradients(zip(gradientes, variables))

nueva_perdida = perdida_fn(lote_y, modelo(lote_x, training=False))
print(f"Pérdida DESPUÉS del paso: {float(nueva_perdida):.4f}")

**Para comentar:** ¿Bajó? ¿Mucho o poco? ¿Qué pasaría si repetimos este paso cientos de veces con todos los datos? Eso es entrenar.

## Paso 4 · Entrena y lee la curva
Construimos un modelo nuevo (desde cero) y lo entrenamos 20 épocas. Guardamos la pérdida y la exactitud en **entrenamiento** y en **validación**.

In [ ]:
modelo = construir_modelo(unidades=16)
historia = modelo.fit(tf.constant(X_entrena), y_entrena,
                      validation_data=(tf.constant(X_valida), y_valida),
                      epochs=20, batch_size=16, verbose=0)
print("Listo.")

In [ ]:
def graficar(historia, titulo="Curva de aprendizaje"):
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
    ejes[0].plot(historia.history["loss"], label="entrenamiento")
    ejes[0].plot(historia.history["val_loss"], label="validación")
    ejes[0].set_title("Pérdida"); ejes[0].set_xlabel("época"); ejes[0].legend()
    ejes[1].plot(historia.history["accuracy"], label="entrenamiento")
    ejes[1].plot(historia.history["val_accuracy"], label="validación")
    ejes[1].set_title("Exactitud"); ejes[1].set_xlabel("época"); ejes[1].legend()
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

graficar(historia)

### Descríbela **antes** de explicarla
Escribe en el chat **solo lo que ves**, sin causas todavía:
- ¿Qué pasa con la pérdida de entrenamiento? ¿Y con la de validación?
- ¿Las dos curvas se parecen o se separan?
- ¿En qué época deja de mejorar la validación?

Solo después pregúntate: ¿**por qué**? Y ¿qué dato necesitarías para saberlo?

## Paso 5 · Predicciones sobre mensajes nuevos
Mensajes que el modelo **no vio**. Incluimos dos difíciles a propósito.

🔮 Escribe en el chat qué categoría esperas **antes** de ejecutar.

In [ ]:
nuevos = [
    ("¿me pueden devolver mi dinero?",                         "pagos"),
    ("el repartidor dejó mi paquete en otro piso",             "envios"),
    ("no logro iniciar sesión desde el celular",               "acceso"),
    ("me cobraron y además no puedo entrar a mi cuenta",       "pagos o acceso (ambiguo)"),
    ("gracias, todo bien",                                      "ninguna de las tres"),
    ("¿a qué hora abre la tienda?",                            "ninguna de las tres"),
]

textos = [t for t, _ in nuevos]
probas = modelo.predict(tf.constant(textos), verbose=0)

for (t, esperada), p in zip(nuevos, probas):
    k = int(np.argmax(p))
    print(f"{t}")
    print(f"   esperada: {esperada:<26} modelo: {CATEGORIAS[k]:<7} (confianza {p[k]:.2f})")

**Para comentar:**
- En los últimos tres, ¿qué hizo el modelo? ¿Tenía una opción correcta?
- ¿Falla **la etiqueta**, **el contexto** o **el modelo**? ¿Qué cambiarías: los datos, las categorías (agregar «otro») o el alcance?
- Una confianza alta, ¿garantiza que la respuesta es correcta?

## Paso 6 · Experimento: pocos datos, modelo más grande
Misma tarea, pero con **solo 24 ejemplos** de entrenamiento, una capa oculta más grande y más épocas.

🔮 ¿Cómo crees que se verá la curva de validación frente a la de entrenamiento?

In [ ]:
X_pocos, y_pocos = X_entrena[:24], y_entrena[:24]

vectorizador = keras.layers.TextVectorization(max_tokens=300, output_sequence_length=12)
vectorizador.adapt(tf.constant(X_pocos))
modelo_grande = keras.Sequential([
    keras.Input(shape=(1,), dtype="string"), vectorizador,
    keras.layers.Embedding(300, 16), keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(len(CATEGORIAS), activation="softmax"),
])
modelo_grande.compile(optimizer=keras.optimizers.Adam(0.01),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])

historia2 = modelo_grande.fit(tf.constant(X_pocos), y_pocos,
                              validation_data=(tf.constant(X_valida), y_valida),
                              epochs=60, batch_size=16, verbose=0)
graficar(historia2, "Pocos datos · modelo más grande")
print(f"Exactitud en entrenamiento: {historia2.history['accuracy'][-1]:.2f}")
print(f"Exactitud en validación:    {historia2.history['val_accuracy'][-1]:.2f}")

### Antes de pedir «más épocas»
Si mejora en entrenamiento pero **no** en validación, revisa primero:
1. **La partición:** ¿validación se parece a lo que verá el modelo en uso?
2. **Los datos:** ¿son suficientes para esta tarea?
3. **La capacidad del modelo:** ¿es demasiado grande para tan pocos ejemplos?

Más épocas casi nunca resuelve esto. Y **no uses el conjunto reservado** para probar ajustes: se queda intacto para el final.

## Para llevarte
Escribe en el chat, en una línea: *¿qué modelo existente usarías para tu tarea y cómo vas a revisar un error que importe?*

Próximo encuentro: cómo se **representa un texto** (tokens y vectores) y una clínica de comparación que también sirve si tu proyecto es de visión.